# 第 1 课：时间窗口与无泄漏划分

目标：把连续三维航迹切成历史输入与未来标签，并验证训练标签不跨入测试段。

In [ ]:
import sys
from pathlib import Path

# 同时兼容：从仓库根目录启动 Jupyter，或从 notebooks/ 目录启动。
search_starts = [Path.cwd(), *Path.cwd().parents]
repo_root = next((path for path in search_starts if (path / "pyproject.toml").exists()), None)
if repo_root is None:
    raise RuntimeError("没有找到 pyproject.toml；请从仓库目录启动 Jupyter。")

src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

print(f"仓库根目录: {repo_root}")
print(f"Python: {sys.version.split()[0]}")

## 1.1 导入并生成一条合成飞行

核心源码：[data.py](../src/memcast_uav/data.py)  
文字讲解：[01_windowing.md](../tutorial/01_windowing.md)

In [ ]:
from memcast_uav.data import make_synthetic_flight, make_train_test_windows

flight = make_synthetic_flight(n_points=360, seed=7)
print("位置数组形状:", flight.positions.shape)
print("采样间隔 dt:", flight.dt)
print("前 5 个意图:", flight.intents[:5])

## 1.2 按时间切分训练与测试窗口

In [ ]:
split_index = 252
history = 24
horizon = 12
stride = 12

train, test = make_train_test_windows(
    flight,
    split_index=split_index,
    history=history,
    horizon=horizon,
    stride=stride,
)
print(f"训练窗口数: {len(train)}，测试窗口数: {len(test)}")

## 1.3 检查第一个测试样本的索引和形状

In [ ]:
first = test[0]
print(
    f"history=[{first.history_start}:{first.forecast_start}), "
    f"future=[{first.forecast_start}:{first.forecast_end})"
)
print("历史/未来形状:", first.history.shape, first.future.shape)
print("意图:", first.intent)

手算：`252 - 24 = 228`，因此历史是 `[228:252)`；未来长度为 12，
所以未来是 `[252:264)`。右端不包含在切片中。

In [ ]:
assert all(window.forecast_end <= split_index for window in train)
assert all(window.forecast_start >= split_index for window in test)
print("无目标泄漏检查通过")

## 1.4 分块练习：把步长改成 6

In [ ]:
train_stride_6, test_stride_6 = make_train_test_windows(
    flight,
    split_index=split_index,
    history=history,
    horizon=horizon,
    stride=6,
)
print("stride=12:", len(train), len(test))
print("stride=6 :", len(train_stride_6), len(test_stride_6))
# TODO：用自己的话解释窗口数量为什么增加，以及相邻未来区间是否重叠。

## 1.5 本课验收

In [ ]:
import subprocess

completed = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_data.py", "-q"],
    cwd=repo_root,
    check=True,
    text=True,
    capture_output=True,
)
print(completed.stdout)

[← 第 0 步](00_setup.ipynb) · [教程目录](README.md) · [下一课：运动学特征 →](02_features.ipynb)